<a href="https://colab.research.google.com/github/m-zayed5722/Miscellaneous-Projects/blob/main/PromptBench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PromptBench: Prompt Benchmarking & Optimization (Colab)

Goal:
Evaluate multiple prompt templates on the same tasks and automatically select the best one.

Pipeline:
Task → Prompt Variants → Generate Outputs → Score → Rank → Select Best Prompt

Features:
- Multiple prompt strategies
- Offline scoring (no API key)
- Prompt leaderboard
- Task-level evaluation
- Extensible to LLM-based evaluation later

In [1]:
#!pip -q install pandas numpy rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.7 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import pandas as pd
from typing import List, Dict
from rapidfuzz import fuzz

In [3]:
TASKS = [
    {
        "id": "t1",
        "input": "Explain RAG in simple terms",
        "reference": "RAG combines retrieval of relevant documents with generation so answers are grounded in external knowledge."
    },
    {
        "id": "t2",
        "input": "Give 3 guardrails for Text-to-SQL",
        "reference": "Use SELECT-only queries, enforce table allowlists, and apply query limits."
    },
    {
        "id": "t3",
        "input": "Why is evaluation important in LLM systems?",
        "reference": "Evaluation helps measure correctness, reliability, and detect issues like hallucination."
    }
]

In [4]:
PROMPTS = {
    "zero_shot": "Answer the following question:\n{input}",

    "structured": """You are an expert assistant.
Provide a clear and structured answer.

Question:
{input}

Answer:""",

    "concise": """Answer in 2-3 concise sentences:
{input}""",

    "step_by_step": """Think step by step and then answer clearly.

Question:
{input}

Steps:
1.

Final Answer:""",

    "bullet_points": """Answer using bullet points:
{input}"""
}

In [5]:
def mock_llm(prompt_name: str, task_input: str) -> str:
    base = task_input.lower()

    if prompt_name == "zero_shot":
        return f"{task_input}. It is important in AI systems."

    elif prompt_name == "structured":
        return f"Answer:\n{task_input} involves key concepts. It is important for reliability and performance."

    elif prompt_name == "concise":
        return f"{task_input} improves system reliability and accuracy."

    elif prompt_name == "step_by_step":
        return f"Step 1: Understand the concept.\nStep 2: Explain clearly.\nFinal Answer: {task_input} ensures better outcomes."

    elif prompt_name == "bullet_points":
        return f"- {task_input}\n- Improves reliability\n- Reduces errors"

    return task_input

In [6]:
def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()

def keyword_overlap(a: str, b: str) -> float:
    a_words = set(re.findall(r"\w+", normalize(a)))
    b_words = set(re.findall(r"\w+", normalize(b)))
    if not a_words or not b_words:
        return 0
    return len(a_words & b_words) / len(a_words | b_words)

def score_output(output: str, reference: str) -> Dict[str, float]:
    sim = keyword_overlap(output, reference)
    fuzz_score = fuzz.token_sort_ratio(output, reference) / 100

    length_penalty = max(0, 1 - abs(len(output) - len(reference)) / 200)

    final = 0.5 * sim + 0.3 * fuzz_score + 0.2 * length_penalty

    return {
        "similarity": round(sim, 3),
        "fuzz": round(fuzz_score, 3),
        "length_score": round(length_penalty, 3),
        "final_score": round(final, 3)
    }

In [7]:
results = []

for task in TASKS:
    for prompt_name, template in PROMPTS.items():
        prompt = template.format(input=task["input"])
        output = mock_llm(prompt_name, task["input"])

        scores = score_output(output, task["reference"])

        results.append({
            "task_id": task["id"],
            "prompt": prompt_name,
            "output": output,
            **scores
        })

df = pd.DataFrame(results)
df

,task_id,prompt,output,similarity,fuzz,length_score,final_score
0,t1,zero_shot,Explain RAG in simple terms. It is important i...,0.087,0.337,0.760,0.297
1,t1,structured,Answer:\nExplain RAG in simple terms involves ...,0.069,0.449,1.000,0.369
2,t1,concise,Explain RAG in simple terms improves system re...,0.087,0.375,0.810,0.318
3,t1,step_by_step,Step 1: Understand the concept.\nStep 2: Expla...,0.067,0.409,0.920,0.340
4,t1,bullet_points,- Explain RAG in simple terms\n- Improves reli...,0.091,0.375,0.810,0.320
5,t2,zero_shot,Give 3 guardrails for Text-to-SQL. It is impor...,0.000,0.317,0.955,0.286
6,t2,structured,Answer:\nGive 3 guardrails for Text-to-SQL inv...,0.037,0.342,0.805,0.282
7,t2,concise,Give 3 guardrails for Text-to-SQL improves sys...,0.045,0.349,0.995,0.326
8,t2,step_by_step,Step 1: Understand the concept.\nStep 2: Expla...,0.000,0.335,0.725,0.245
9,t2,bullet_points,- Give 3 guardrails for Text-to-SQL\n- Improve...,0.000,0.322,0.995,0.296
